In [ ]:
!pip install ultralytics
from IPython import display
display.clear_output()

from ultralytics import YOLO
from IPython.display import display, Image
from google.colab import drive
drive.mount('/content/drive')

import time
import shutil
import os
import pickle
import sys
from tqdm import tqdm

Mounted at /content/drive


In [ ]:

# Append your project directory to sys.path
sys.path.append('/content/drive/My Drive/Projects/chromis/Bommie_feeding')
from cut_chromis_track_vids import process_video

# Directory where videos are stored
video_folder = "/content/drive/My Drive/Projects/chromis/Bommie_feeding/Deployments/30_11_2023/7"
#video_folder = "/content/drive/My Drive/Projects/chromis/Bommie_feeding/test"
out_folder_path = os.path.join(video_folder, "output")
os.makedirs(out_folder_path, exist_ok=True)

# Instantiate the YOLO models
model = YOLO('drive/My Drive/Projects/chromis/Bommie_feeding/Close_chomis_yolo_data/YoloCheckPoints/train10/weights/best.pt')
bite_model =  YOLO("/content/drive/My Drive/bite_medium_march_13.pt")

In [ ]:
import cv2
def extract_and_process_frames(video_path, model, batch_si  ze=128):
    # Load the video
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print("Error: Could not open video.")
        return

    frames = []
    results = []
    while True:
        ret, frame = cap.read()
        if not ret:
            break  # No more frames or error
        frames.append(frame)
        # Process in batches
        if len(frames) == batch_size:
            batch_results = model.predict(source=frames,verbose=False)  # Perform inference on the batch
            results.extend(batch_results)
            frames = []  # Reset for next batch

    # Process remaining frames if there are any
    if frames:
        batch_results = model.predict(source=frames)
        results.extend(batch_results)

    cap.release()
    return results

In [ ]:
start = time.time()


# Process each video in the folder
# Process each video in the folder
for video_filename in os.listdir(video_folder):
    if video_filename.endswith(".mp4"):
        video_path = os.path.join(video_folder, video_filename)
        video_name = os.path.splitext(video_filename)[0]
        print(video_path)



        # Run YOLO model on the video
        results = model.track(source=video_path, stream=True)

        # Process tracking results
        all_bboxes = []
        for result in results:
            frame_data = []
         #   if result.boxes is None or result.boxes.id is None:

            if result.boxes is not None and result.boxes.id is not None:
                for bbox, track_id in zip(result.boxes.xyxy, result.boxes.id):
                    bbox = [bbox[0].item(), bbox[1].item(), bbox[2].item(), bbox[3].item()]  # Convert bbox tensor to list
                    frame_data.append({"track_id": track_id.item(), "bbox": bbox})
            else:
                frame_data.append({"track_id": None, "bbox": None})
            all_bboxes.append(frame_data)

        # Interpolate bboxes
        #interpolate_bboxes(all_bboxes)

        # Save annotations as a pkl file
        # out_file_name = video_name
        new_file_path = out_folder_path + f"/{video_name}.pkl"
        with open(new_file_path, 'wb') as f:
            pickle.dump(all_bboxes, f)

        first_appearances = {}

        for frame_index, frame in enumerate(all_bboxes):
            for track in frame:
                track_id = track['track_id']
                # If this track_id has not been seen before, record its first appearance
                if track_id not in first_appearances:
                    first_appearances[track_id] = frame_index

        # Process videos to generate cut out videos
        clip_dir = "/content/output/"

        #clear clips if exisit
        if os.path.exists(clip_dir):
          shutil.rmtree('/content/output')

        os.makedirs(clip_dir, exist_ok=True)

        process_video(video_path, new_file_path, extract_dim=(220, 220), n_pixels_crop=2, output_dir=clip_dir)

        #save zip folder of clips
        zip_dir = "/content/output.zip"
        !zip -r {zip_dir} {clip_dir}
        out_zip_dir = os.path.join(out_folder_path) + f"/{video_name}.zip"
        shutil.move(zip_dir, out_zip_dir)


        output_filename = video_name + ".txt"
        output_path = os.path.join(out_folder_path, output_filename)

                # Process videos and write results to one file
        # Process videos and write results to one file
        with open(output_path, 'w') as file:
            for video_file in os.listdir(clip_dir):
                if video_file.endswith(".mp4"):
                    video_path = os.path.join(clip_dir, video_file)
                    results = extract_and_process_frames(video_path, bite_model)

                    for i, result in tqdm(enumerate(results, 1), total=len(results), desc=f"Processing {video_file}"):
                        track_id = int(video_file.split('_')[1].split('.')[0])
                        first_frame = first_appearances.get(track_id, 0)
                        global_frame_number = i + first_frame
                        max_prob_class = result.probs.data.argmax()

                        # Initialize variables to store mean_x, mean_y, and bb_area
                        mean_x, mean_y, bb_area = 'NA', 'NA', 'NA'

                        # Check if the current frame is within the range of all_bboxes
                        if global_frame_number < len(all_bboxes):
                            # Check if the current frame has data and if it matches the track_id
                            frame_data = all_bboxes[global_frame_number]
                            if frame_data:
                                for bbox_data in frame_data:
                                    if bbox_data and bbox_data['track_id'] == track_id:
                                        x1, y1, x2, y2 = map(float, bbox_data["bbox"])
                                        bb_area= (x2 - x1)*(y2 - y1)
                                        mean_x = (x2 + x1) /2
                                        mean_y = (y2 + y1) /2
                                        break

                        # Write track_id, frame, class, mean_x, mean_y, and bb_area to the file
                        file.write(f"{track_id}, {global_frame_number+1}, {max_prob_class}, {mean_x}, {mean_y}, {bb_area}\n")

        end = time.time()

print (end - start)